# Policy retrieval, measured

ArchCompass judges a detected candidate against the policies a retriever put in front of
the model. Everything downstream — the verdict, the evidence, the questions the run comes
back with — is bounded by that selection: a policy that was not retrieved cannot be
weighed, and no amount of reasoning recovers it.

This notebook measures that step against the corpus the product actually ships, with the
embedding model it actually runs locally: **`embeddinggemma` on Ollama, 768 dimensions**,
built through `archcompass.reasoning.adapters.factory.build_embeddings`, indexed by the
production chunker, ranked by the production cosine query.

It answers four questions.

1. **How good is the ranking?** Recall, precision, MRR, MAP and nDCG over a hand-labelled
   test set of 68 situations.
2. **Compared to what?** A BM25 lexical baseline and a random floor, run through the same
   port so nothing but the ranking differs.
3. **Do the design choices earn their place?** Heading chunks against whole documents,
   EmbeddingGemma's task prompts against none, and the case constraints against a query
   without them.
4. **Does the shipped configuration pass its own release gate?** The gate in
   `docs/policy-retrieval.md`, applied to real runs at K = 8, 12, 16 and 20.

**Running it.** Needs Ollama on `localhost:11434` holding `embeddinggemma`
(`ollama pull embeddinggemma`), and the notebook dependency group:

```
uv sync --group evaluation
uv run --group evaluation jupyter lab evaluation/retrieval-evaluation.ipynb
```

or headless, which is what `make eval-rag` does:

```
make eval-rag
```

No reasoning model is used and no network beyond the local Ollama is reached.

In [ ]:
from __future__ import annotations

import json
import sqlite3
import sys
import time
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import pandas as pd
import yaml

# The repository root, found rather than assumed: Jupyter's working directory depends on
# where it was launched from, and every path below is relative to the root.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evaluation.harness import (  # noqa: E402
    Bm25PolicyIndex,
    InMemoryDenseIndex,
    RandomPolicyIndex,
    TaskPrefixedEmbeddings,
    candidate_cases,
    chunk_report,
    evaluation_corpus,
    gate_examples,
    heading_chunks,
    intent_cases,
    label_coverage,
    load_cases,
    ollama_embeddings,
    run_index,
    run_retriever,
    score_case,
    scoped_policies,
    shipped_corpus,
    summarize,
    whole_document_chunks,
)

EMBEDDING_MODEL = "embeddinggemma"
EMBEDDING_DIMENSIONS = 768
OLLAMA = "http://localhost:11434"

#: Reported at every one of these. 20 is the shipped `DENSE_RETRIEVER_RELEASE_TOP_K`; the
#: smaller values are where a retriever that is merely *eventually* right stops looking
#: right, and they are the values the release gate evaluates.
KS = (1, 3, 5, 8, 10, 12, 16, 20)

RESULTS = ROOT / "evaluation" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# Fail here, with a sentence, rather than three cells later inside a retry loop.
try:
    tags = httpx.get(f"{OLLAMA}/api/tags", timeout=5).json()
except httpx.HTTPError as error:  # pragma: no cover - notebook preflight
    raise SystemExit(f"Ollama is not answering on {OLLAMA}: {error}") from error
installed = {model["name"].split(":")[0] for model in tags["models"]}
if EMBEDDING_MODEL not in installed:
    raise SystemExit(f"`ollama pull {EMBEDDING_MODEL}` first; this host has {sorted(installed)}")

print(f"root                {ROOT}")
print(f"embedding           ollama:{EMBEDDING_MODEL}:{EMBEDDING_DIMENSIONS}")
print(f"ollama models       {', '.join(sorted(installed))}")

## 1. The corpus

The shipped general policies, read by the same parser a workspace reads them with. A policy
that has drifted out of the nine-heading format fails this cell rather than quietly
producing a worse embedding.

The index does not hold policies; it holds **chunks**. `SQLitePolicyIndex` splits each
document at its `## ` headings and prefixes every piece with the policy title, so one
policy becomes nine vectors and a query matches the *section* that answers it. A policy's
score is the best of its chunks.

In [ ]:
corpus = shipped_corpus(ROOT)
scope_fixture = scoped_policies(ROOT)
full_corpus = evaluation_corpus(ROOT)

headings = chunk_report(corpus, heading_chunks)
documents = chunk_report(corpus, whole_document_chunks)

print(f"shipped policies         {headings.policies}")
print(f"heading chunks           {headings.chunks} ({headings.chunks_per_policy:.1f} per policy)")
print(f"longest heading chunk    {headings.longest_characters} chars "
      f"(~{headings.longest_estimated_tokens} tokens, {headings.longest_chunk_policy})")
print(f"longest whole document   {documents.longest_characters} chars "
      f"(~{documents.longest_estimated_tokens} tokens)")
print(f"EmbeddingGemma context   2048 tokens on this Ollama build")
print()
print(f"scope fixture            {len(scope_fixture)} policies, for the mandatory arm in section 8")
for policy in scope_fixture:
    print(f"  {policy.id:32s} {policy.scope.value:14s} {policy.strength.value:10s} -> {policy.applies_to}")
print()
print("--- one chunk, as the index stores it ---")
print(heading_chunks(corpus[0])[1])

## 2. The test set

68 labelled situations, of two kinds.

**28 candidate cases.** Not written by hand: the shipped example repositories under
`eval/cases` are parsed by `PythonAstRepositoryAnalyzer`, run through
`detect_finding_candidates`, and turned into the query by `retrieval_query` — the
production path end to end. `evaluation/dataset/candidate-labels.yaml` holds only the
relevance labels, joined to the detector output by participant list. An unmatched label or
an unlabelled candidate fails the load, so a detector change cannot silently shrink the
test set.

**40 intent cases.** Design situations described the way an engineer describes them, in
`evaluation/dataset/intent-cases.yaml`. They exist because the detectors know three shapes,
and a test set built from those alone would measure the corpus for three questions and say
nothing about the other fifty policies. Each is written from the *problem's* vocabulary,
not the policy's — a retriever that only works when the query quotes the document is a
keyword search with extra steps.

**Grades.** `bearing` is what a reviewer would cite in the verdict; missing one is a wrong
retrieval, and recall is reported over bearings alone. `supporting` and `adjacent` are
relevant without being load-bearing, and only nDCG reads them (gains 3 / 2 / 1).

In [ ]:
cases = load_cases(ROOT)
by_id = {case.id: case for case in cases}

named, never_labelled, unknown = label_coverage(cases, full_corpus)
if unknown:
    raise SystemExit(f"Labels name policies that do not exist: {sorted(unknown)}")

frame = pd.DataFrame(
    [
        {
            "case": case.id,
            "kind": case.kind,
            "pattern": case.pattern,
            "bearing": len(case.bearing),
            "graded": len(case.grades),
            "query chars": len(case.query),
        }
        for case in cases
    ]
)
print(f"cases                    {len(cases)}  "
      f"({(frame['kind'] == 'candidate').sum()} candidate, {(frame['kind'] == 'intent').sum()} intent)")
print(f"bearing policies / case  {frame['bearing'].mean():.2f} (min {frame['bearing'].min()}, "
      f"max {frame['bearing'].max()})")
print(f"graded policies / case   {frame['graded'].mean():.2f}")
print(f"corpus policies labelled {len(named)}/{len(corpus)} of the shipped corpus")
print(f"never labelled           {sorted(never_labelled) or 'none'}")
print()
display(frame.groupby(["kind", "pattern"]).agg(
    cases=("case", "count"), bearing=("bearing", "mean"), graded=("graded", "mean")
).round(2))
print()
print("--- one candidate query, verbatim ---")
print(cases[0].query)
print()
print("bearing:", sorted(cases[0].bearing))

## 3. The index, and a parity check

Two indexes are built. `SQLitePolicyIndex` is the shipped one — sqlite-vec, content-hashed
chunks, cosine distance in SQL. `InMemoryDenseIndex` is the harness's, which exists because
chunking is a variable here and a constant there, and because an ablation sweep should not
re-embed a corpus it has already embedded.

A metric from the second one is only worth reading if the two agree, so they are checked
against each other on every query in the test set before anything else is measured.

In [ ]:
embeddings = ollama_embeddings(
    model=EMBEDDING_MODEL, dimensions=EMBEDDING_DIMENSIONS, base_url=OLLAMA
)
identity = f"ollama:{EMBEDDING_MODEL}:{EMBEDDING_DIMENSIONS}"

from archcompass.policies.adapters import SQLitePolicyIndex  # noqa: E402

database = RESULTS / "policy-index.sqlite3"
production = SQLitePolicyIndex(
    lambda: sqlite3.connect(database),
    embeddings,
    embedding_identity=identity,
    dimensions=EMBEDDING_DIMENSIONS,
)

started = time.perf_counter()
production.synchronize(corpus)
indexing = time.perf_counter() - started
print(f"SQLitePolicyIndex.synchronize  {indexing:6.1f}s  "
      f"({'embedded {} chunks'.format(headings.chunks) if indexing > 5 else 'already indexed'})")

memory = InMemoryDenseIndex(
    embeddings, embedding_identity=identity, dimensions=EMBEDDING_DIMENSIONS
)
started = time.perf_counter()
memory.synchronize(corpus)
print(f"InMemoryDenseIndex.synchronize {time.perf_counter() - started:6.1f}s")

disagreements = []
for case in cases:
    left = tuple(match.policy_id for match in production.search(case.query, limit=20))
    right = tuple(match.policy_id for match in memory.search(case.query, limit=20))
    if left != right:
        disagreements.append((case.id, left, right))

print()
if disagreements:
    for case_id, left, right in disagreements[:5]:
        print(f"DISAGREES {case_id}\n  sqlite {left}\n  memory {right}")
    raise SystemExit(f"{len(disagreements)} of {len(cases)} rankings differ; the ablations below "
                     "would not be measuring the shipped retriever")
print(f"parity: the two indexes return identical top-20 rankings on all {len(cases)} queries")

## 4. Ranking quality

Every metric here reads the **bearing** set except nDCG, which reads the grades.

- **recall@k** — the share of bearing policies inside the first k. The one that matters:
  a bearing policy outside the window cannot be weighed at all.
- **precision@k** — bounded above by `bearings / k`, so it falls as k rises by construction.
  Read as a budget number, not a quality number.
- **MRR** — 1/rank of the first bearing hit.
- **MAP** — precision at each hit, averaged. Rewards getting *all* the bearings early.
- **nDCG@k** — graded gain, discounted by rank, against the best ordering the grades allow.
  The only metric that separates "found it at rank 1" from "found it at rank 18".

All macro averages: every case counts once, whatever the size of its bearing set.

In [ ]:
def score_all(runs, label):
    scores = [
        score_case(
            case_id=run.case_id,
            kind=run.kind,
            pattern=run.pattern,
            ranked=run.ranked,
            bearing=by_id[run.case_id].bearing,
            grades=by_id[run.case_id].grades,
            ks=KS,
        )
        for run in runs
    ]
    return summarize(label, scores, KS), scores


def as_row(summary, extra=None):
    row = {"variant": summary.label, "cases": summary.cases}
    row |= {f"R@{k}": value for k, value in summary.recall_at_k}
    row |= {"MRR": summary.mrr, "MAP": summary.map_score,
            "nDCG@10": summary.ndcg_at_10, "nDCG@20": summary.ndcg_at_20,
            "complete@20": summary.complete_at_20}
    return row | (extra or {})


dense_runs = run_index(memory, corpus, cases, limit=20)
dense, dense_scores = score_all(dense_runs, f"{EMBEDDING_MODEL} (shipped)")

display(pd.DataFrame([as_row(dense)]).set_index("variant").round(3))
print()
print("recall@20 by kind:   ", {kind: round(value, 3) for kind, value in dense.recall_by_kind})
print("recall@20 by pattern:", {pattern: round(value, 3) for pattern, value in dense.recall_by_pattern})
print()
print(f"query latency: median {pd.Series([r.seconds for r in dense_runs]).median() * 1000:.0f} ms, "
      f"p95 {pd.Series([r.seconds for r in dense_runs]).quantile(0.95) * 1000:.0f} ms "
      "(one embed call plus an exact scan of 486 vectors)")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].plot(KS, [dense.recall(k) for k in KS], marker="o", label="recall@k (bearing)")
axes[0].plot(KS, [dense.precision(k) for k in KS], marker="s", label="precision@k")
axes[0].axvline(20, color="grey", linestyle=":", linewidth=1)
axes[0].annotate("shipped K", (20, 0.05), ha="right", fontsize=8, color="grey")
axes[0].set_xlabel("k"), axes[0].set_ylabel("score"), axes[0].set_ylim(0, 1.02)
axes[0].set_title("Where the bearing policies land"), axes[0].legend(fontsize=8)

patterns = dict(dense.recall_by_pattern)
order = sorted(patterns, key=patterns.get)
bars = axes[1].barh(order, [patterns[name] for name in order], color="#4c72b0")
axes[1].axvline(0.90, color="crimson", linestyle="--", linewidth=1)
axes[1].annotate("gate floor 0.90", (0.90, -0.45), fontsize=8, color="crimson", ha="center")
axes[1].set_xlim(0, 1.02), axes[1].set_xlabel("recall@20")
axes[1].set_title("Recall@20 by pattern")
axes[1].bar_label(bars, fmt="%.2f", fontsize=8, padding=3)
figure.tight_layout()
plt.show()

## 5. Where it fails

An aggregate says how much is missed, never what. This is every case with a bearing policy
outside the top 20, and the rank of each bearing policy for the rest — because a policy
found at rank 19 is a near miss the next corpus edit can turn into a real one.

In [ ]:
missed = pd.DataFrame(
    [
        {
            "case": score.case_id,
            "pattern": score.pattern,
            "missed at 20": ", ".join(score.missed_at_20),
            "R@20": score.recall(20),
        }
        for score in dense_scores
        if score.missed_at_20
    ]
).sort_values(["pattern", "case"])

print(f"{len(missed)} of {len(cases)} cases miss at least one bearing policy inside K=20")
display(missed.set_index("case"))

print()
print("Rank of every bearing policy, worst first:")
ranks = pd.DataFrame(
    [
        {"case": score.case_id, "pattern": score.pattern, "policy": policy,
         "rank": rank if rank else 999}
        for score in dense_scores
        for policy, rank in score.bearing_ranks
    ]
)
display(ranks.sort_values("rank", ascending=False).head(18).set_index("case"))

print()
print("Bearing policies missed most often across the whole set:")
display(
    ranks[ranks["rank"] > 20]["policy"].value_counts().rename("times missed").to_frame()
)

## 6. Baselines

A recall figure alone is unreadable. 0.90 is excellent against a floor of 0.35 and
unremarkable against a keyword search that reaches 0.85 for nothing.

Both baselines satisfy `DensePolicyIndex`, so they are driven by exactly the code that
drives the embeddings, over exactly the same chunks, with the same best-chunk-wins
reduction. What differs is only the ranking function.

- **BM25** — Okapi over the same chunks. Deliberately untuned: a baseline adjusted until it
  loses stops being a baseline.
- **Random** — a deterministic permutation per query. The floor: with 54 policies and about
  two bearings each, drawing 20 finds roughly a third of them by luck alone.

In [ ]:
variants = {}
variants[dense.label] = (dense, dense_scores)

lexical_runs = run_index(Bm25PolicyIndex(), corpus, cases, limit=20)
variants["BM25 (lexical baseline)"] = score_all(lexical_runs, "BM25 (lexical baseline)")

random_runs = run_index(RandomPolicyIndex(), corpus, cases, limit=20)
variants["random (floor)"] = score_all(random_runs, "random (floor)")

comparison = pd.DataFrame([as_row(summary) for summary, _ in variants.values()])
display(comparison.set_index("variant").round(3))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))

for label, (summary, _) in variants.items():
    axes[0].plot(KS, [summary.recall(k) for k in KS], marker="o", label=label)
axes[0].set_xlabel("k"), axes[0].set_ylabel("recall@k (bearing)"), axes[0].set_ylim(0, 1.02)
axes[0].set_title("Embeddings against a keyword search and luck"), axes[0].legend(fontsize=8)

labels = list(variants)
width = 0.26
positions = range(len(["MRR", "MAP", "nDCG@10"]))
for offset, label in enumerate(labels):
    summary = variants[label][0]
    values = [summary.mrr, summary.map_score, summary.ndcg_at_10]
    axes[1].bar([p + offset * width for p in positions], values, width, label=label)
axes[1].set_xticks([p + width for p in positions])
axes[1].set_xticklabels(["MRR", "MAP", "nDCG@10"])
axes[1].set_ylim(0, 1.0), axes[1].set_title("Rank-sensitive metrics")
axes[1].legend(fontsize=8)
figure.tight_layout()
plt.show()

## 7. Ablations

Three choices the shipped retriever makes, each measured against its alternative. The
embedding cache is shared, so a variant that reuses a text does not pay for it twice.

**Task prompts.** EmbeddingGemma is trained with task prefixes —
`task: search result | query: …` for a query, `title: … | text: …` for a document. Ollama's
modelfile for it is a bare `{{ .Prompt }}`, and `build_embeddings` adds nothing, so the
shipped path uses none of them. Whether that costs anything is a measurement, not an
opinion.

**Chunking.** The production index splits at `## ` headings — nine vectors per policy
instead of one. It is more storage, more embed calls and a longer sync. The question is
whether matching a *section* beats matching a whole document.

**Case constraints.** `retrieval_query` appends the case's constraints to every query. It
is the only part of the query that is not derived from the candidate, and it is the part a
user writes.

In [ ]:
prefixed = TaskPrefixedEmbeddings(embeddings)
ablations = {}
ablations["heading chunks, no task prompts (shipped)"] = (dense, dense_scores)

task_index = InMemoryDenseIndex(
    prefixed, embedding_identity=f"{identity}:prompted", dimensions=EMBEDDING_DIMENSIONS
)
ablations["heading chunks + task prompts"] = score_all(
    run_index(task_index, corpus, cases, limit=20), "heading chunks + task prompts"
)

document_index = InMemoryDenseIndex(
    embeddings, embedding_identity=f"{identity}:whole", dimensions=EMBEDDING_DIMENSIONS,
    chunker=whole_document_chunks,
)
ablations["whole documents, no task prompts"] = score_all(
    run_index(document_index, corpus, cases, limit=20), "whole documents, no task prompts"
)

display(pd.DataFrame([as_row(summary) for summary, _ in ablations.values()])
        .set_index("variant").round(3))

In [ ]:
# The constraint ablation runs over the 28 candidate cases only — intent queries have no
# case to strip — so it is scored on its own and never mixed into the table above.
with_constraints = candidate_cases(ROOT, with_constraints=True)
without_constraints = candidate_cases(ROOT, with_constraints=False)


def score_subset(subset, label):
    lookup = {case.id: case for case in subset}
    index = InMemoryDenseIndex(
        embeddings, embedding_identity=identity, dimensions=EMBEDDING_DIMENSIONS
    )
    runs = run_index(index, corpus, subset, limit=20)
    scores = [
        score_case(
            case_id=run.case_id, kind=run.kind, pattern=run.pattern, ranked=run.ranked,
            bearing=lookup[run.case_id].bearing, grades=lookup[run.case_id].grades, ks=KS,
        )
        for run in runs
    ]
    return summarize(label, scores, KS)


constraints = pd.DataFrame([
    as_row(score_subset(with_constraints, "candidate query with case constraints")),
    as_row(score_subset(without_constraints, "candidate query without case constraints")),
])
display(constraints.set_index("variant").round(3))

## 8. The release gate

`docs/policy-retrieval.md` states the gate a retriever must pass before its K and version
constants move, and `archcompass.policies.evaluation` implements it. This section runs the
**whole shipped retriever** — `DensePolicyRetriever`, mandatory merge and provenance
included — at each evaluated K and applies that gate to the result.

The corpus here is the shipped 54 plus the four scoped fixtures, because the mandatory arm
has nothing to include otherwise: every bundled policy is general guidance, and the rule
being tested is that an applicable organisation, repository or required policy is included
whatever the embeddings say.

One gate is not covered. `verdict_regression` compares the verdict a judge reaches on the
retrieved policies against the verdict it reaches on the full corpus, which costs a
reasoning model per candidate per K. It is reported as uncovered below rather than assumed
to pass.

In [ ]:
from archcompass.policies.evaluation import choose_smallest_passing_k, evaluate_retrieval  # noqa: E402

gate_index = InMemoryDenseIndex(
    embeddings, embedding_identity=f"{identity}:gate", dimensions=EMBEDDING_DIMENSIONS
)
gate_index.synchronize(full_corpus)

per_k = {}
rows = []
for top_k in (8, 12, 16, 20):
    runs = run_retriever(gate_index, full_corpus, with_constraints, top_k=top_k)
    examples = gate_examples(runs, full_corpus)
    per_k[top_k] = examples
    result = evaluate_retrieval(examples)
    rows.append({
        "K": top_k,
        "macro recall": result.macro_recall,
        "worst pattern recall": min(value for _, value in result.recall_by_pattern),
        "complete coverage": result.complete_coverage,
        "mandatory coverage": result.mandatory_coverage,
        "verdict regression": result.verdict_regression,
        "passes": result.passed,
    })

gate = pd.DataFrame(rows).set_index("K")
display(gate.round(3))

print()
print("gate floors: macro recall >= 0.95, every pattern >= 0.90, complete coverage >= 0.75,")
print("             mandatory coverage == 1.00, verdict regression <= 0.10")
print()
try:
    chosen, result = choose_smallest_passing_k(lambda k: per_k[k])
    print(f"PASS - smallest passing K is {chosen} (macro recall {result.macro_recall:.3f})")
except ValueError as refusal:
    print(f"FAIL - {refusal}")
print()
print("verdict_regression reads 0.000 because it is UNCOVERED, not because it was measured:")
print("no reference run against the full corpus was judged. Treat that column as unknown.")

In [ ]:
# The mandatory arm on its own. Every scoped or required policy that applies to a case must
# be selected whatever the ranking says, and one that does not apply must never appear.
checks = []
for case, retrieved in run_retriever(gate_index, full_corpus, with_constraints, top_k=20):
    selected = {policy.id for policy in retrieved.policies}
    context = case.case.policy_context
    for policy in scope_fixture:
        applies = policy.applies_in(
            user=context.user, organisation=context.organisation, repository=context.repository
        )
        expected = applies and (policy.scope.value != "general" or policy.strength.value == "required")
        checks.append({
            "repository": case.repository,
            "policy": policy.id,
            "applies": applies,
            "must be selected": expected,
            "selected": policy.id in selected,
            "ok": (policy.id in selected) == expected,
        })

scope_frame = pd.DataFrame(checks)
summary = scope_frame.groupby(["repository", "policy"]).agg(
    must=("must be selected", "first"), selected=("selected", "first"), ok=("ok", "all")
)
display(summary)
print()
print(f"mandatory arm: {int(scope_frame['ok'].sum())}/{len(scope_frame)} checks correct "
      f"across {scope_frame['repository'].nunique()} cases")
if not scope_frame["ok"].all():
    display(scope_frame[~scope_frame["ok"]])

## 9. Cost

What the retrieval step costs on this machine, so the numbers above can be weighed against
the wait a user actually sits through.

In [ ]:
sample = [case.query for case in cases[:20]]
started = time.perf_counter()
for query in sample:
    embeddings.embed_query(query)
query_embed = (time.perf_counter() - started) / len(sample)

started = time.perf_counter()
embeddings.embed_documents(heading_chunks(corpus[0]) * 8)
document_embed = (time.perf_counter() - started) / (len(heading_chunks(corpus[0])) * 8)

latency = pd.Series([run.seconds for run in dense_runs])
print(f"first index build (486 chunks)   {indexing:6.1f} s   once per corpus edit, per model")
print(f"one query embedding              {query_embed * 1000:6.0f} ms  uncached, on the request path")
print(f"one document embedding           {document_embed * 1000:6.0f} ms  batched during indexing")
print(f"search after embedding, median   {latency.median() * 1000:6.0f} ms  exact scan over 486 vectors")
print(f"search after embedding, p95      {latency.quantile(0.95) * 1000:6.0f} ms")
print()
print(f"a review of a repository with 8 candidates spends about "
      f"{8 * (query_embed + latency.median()):.1f}s in retrieval, once the index is warm.")

## 10. Results on disk

Three files. The first two are the evidence behind every table above. The third is the
input `archcompass retrieval evaluate --from` reads, which is the release gate's own entry
point — so the gate can be re-checked from the command line without opening a notebook.

In [ ]:
rows = []
for label, (_, scores) in {**variants, **ablations}.items():
    for score in scores:
        rows.append({
            "variant": label, "case": score.case_id, "kind": score.kind,
            "pattern": score.pattern,
            **{f"R@{k}": value for k, value in score.bearing_recall_at_k},
            "MRR": score.reciprocal_rank, "AP": score.average_precision,
            "nDCG@10": score.ndcg_at_10, "complete@20": score.complete_at_20,
            "missed@20": ", ".join(score.missed_at_20),
        })
case_scores = pd.DataFrame(rows)
case_scores.to_csv(RESULTS / "case-scores.csv", index=False)

variant_summary = pd.DataFrame(
    [as_row(summary) for summary, _ in {**variants, **ablations}.values()]
)
variant_summary.to_csv(RESULTS / "variant-summary.csv", index=False)

(RESULTS / "evaluation.yaml").write_text(
    yaml.safe_dump(
        {
            "embedding_identity": identity,
            "results": {
                top_k: [
                    {
                        "pattern": example.pattern,
                        "expected_policy_ids": sorted(example.expected_policy_ids),
                        "selected_policy_ids": list(example.selected_policy_ids),
                        "required_policy_ids": sorted(example.required_policy_ids),
                        "scoped_policy_ids": sorted(example.scoped_policy_ids),
                    }
                    for example in examples
                ]
                for top_k, examples in per_k.items()
            },
        },
        sort_keys=False,
    ),
    encoding="utf-8",
)

(RESULTS / "summary.json").write_text(
    json.dumps(
        {
            "embedding_identity": identity,
            "corpus_policies": len(corpus),
            "corpus_chunks": headings.chunks,
            "cases": len(cases),
            "shipped": {
                "recall_at_k": dict(dense.recall_at_k),
                "mrr": dense.mrr,
                "map": dense.map_score,
                "ndcg_at_10": dense.ndcg_at_10,
                "complete_at_20": dense.complete_at_20,
                "recall_by_pattern": dict(dense.recall_by_pattern),
            },
            "gate": gate.reset_index().to_dict(orient="records"),
            "index_build_seconds": indexing,
            "query_embed_seconds": query_embed,
        },
        indent=2,
    ),
    encoding="utf-8",
)

for path in sorted(RESULTS.glob("*")):
    if path.suffix != ".sqlite3":
        print(f"{path.relative_to(ROOT)}  {path.stat().st_size:,} bytes")
print()
print("re-check the gate from a shell with:")
print("  uv run archcompass retrieval evaluate --from evaluation/results/evaluation.yaml")

PLACEHOLDER_FINDINGS